In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib import animation

from IPython.display import HTML

import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

import os
import cmocean

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as ctx

import glob
from PIL import Image
import imageio.v3 as iio
import scienceplots

In [ ]:
# Cell-0-Spinup
base_dir = "/home/jovyan/Cloud Storage/naa-vre-waddenzee-shared/dws/model_output/archived_runs"
spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])

In [ ]:
# Cell-1-frames
# ------------------------------ Helpers -----------------------------------

def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(
            f"Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}"
        )
    return remain[0], remain[1]
    
frame_counter = 0   # global counter for MP4 ordering
def _find_time_dim(da: xr.DataArray) -> str:
    for dim in da.dims:
        if "time" in dim.lower():
            return dim
    raise ValueError(f"No time-like dimension found in {da.dims}")

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str:
    candidates = ("z", "sigma", "layer", "level", "lev", "depth", "nmesh2_layer_3d")
    for dim in da.dims:
        if dim == time_dim:
            continue
        if any(key in dim.lower() for key in candidates):
            return dim
    # Fallback: assume second dimension is vertical for 4D fields [time, z, y, x]
    if len(da.dims) >= 3:
        return da.dims[1]
    raise ValueError(f"No vertical-like dimension found in {da.dims}")

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _pick_existing_var(ds: xr.Dataset, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"Could not find any of these variables: {candidates}")

def _find_horizontal_dims(da: xr.DataArray, exclude: tuple[str, ...] = ()) -> tuple[str, str]:
    dims = [dim for dim in da.dims if dim not in exclude]
    if len(dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims, got {dims} from {da.dims}")
    return dims[0], dims[1]

def _compute_edges(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        raise ValueError("Need at least two points to build cell edges.")
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0]    = values[0]  - 0.5 * (values[1]  - values[0])
    edges[-1]   = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    """Remove duplicate timestamps along the time dimension."""
    time_values = da[time_dim].values
    _, unique_idx = np.unique(time_values, return_index=True)
    unique_idx = np.sort(unique_idx)
    return da.isel({time_dim: unique_idx})

def extract_transect(
    ds: xr.Dataset,
    var_name: str,
    i_lon: int,
    lat_start: int,
    lat_stop: int,
    lon_name_candidates: tuple[str, ...] = ("lonc", "lon", "longitude"),
    lat_name_candidates: tuple[str, ...] = ("latc", "lat", "latitude"),
) -> dict[str, xr.DataArray]:
    da = ds[var_name]
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    y_dim, x_dim = _find_xy_dims(da, time_dim, z_dim)

    transect = da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})

    lat_name = _pick_coord_name(ds, lat_name_candidates)
    if lat_name is None:
        raise KeyError(f"Could not find latitude variable among {lat_name_candidates}")

    lat_da = ds[lat_name]
    if y_dim in lat_da.dims and x_dim in lat_da.dims:
        lat_line = lat_da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
    elif y_dim in lat_da.dims:
        lat_line = lat_da.isel({y_dim: slice(lat_start, lat_stop)})
    else:
        raise ValueError(f"Latitude variable '{lat_name}' does not match transect dims")

    if "h" in ds.variables:
        h = ds["h"].isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
        if z_dim in h.dims:
            depth = -h.cumsum(dim=z_dim)
        else:
            depth = None
    else:
        depth = None

    return {
        "transect": transect,
        "lat_line": lat_line,
        "depth": depth,
        "time_dim": transect.dims[0],
        "z_dim": z_dim,
        "y_dim": y_dim,
    }


# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

base_dir = "/home/jovyan/Cloud Storage/naa-vre-waddenzee-shared/dws/model_output/archived_runs"
frames_root = "/home/jovyan/Cloud Storage/naa-vre-waddenzee-shared/dws/results/animation/pelagic"
os.makedirs(frames_root, exist_ok=True)

#variable    = "O2o"         # <--- your requested variable
#label       = "Oxygen"
#var_unit = "mmolO2 m$^{-3}$"
#VMIN = 0
#VMAX = 400
#CMAP = cmocean.cm.oxy

# variable    = "salt"         # <--- your requested variable
#label       = "Salinity (psu)"
#var_unit = "psu"
#VMIN = 0
#VMAX = 42
#CMAP = cmocean.cm.haline

# variable    = "temp"         
# label       = "Temperature (degrees)"
#VMIN = 0
#VMAX = 22
#CMAP = cmocean.cm.thermal

#variable = "P1c" 
#label = "Diatoms (P1c)"
#var_unit  = "mgC m$^{-3}$"
#VMIN = 0
#VMAX = 4000
#CMAP = cmocean.cm.algae

#variable = "P2c" 
#label = "Flagellates (P2c)"
#var_unit  = "mgC m$^{-3}$"
#VMIN = 0
#VMAX = 300
#CMAP = cmocean.cm.algae

variable = "P3c" 
label = "PicoPhytoPlankton (P3c)"
var_unit  = "mgC m$^{-3}$"
VMIN = 0
VMAX = 600
CMAP = cmocean.cm.algae

#TOPVIEW_DIR  = Path(frames_root) / variable / "topview"
#TRANSECT_DIR = Path(frames_root) / variable / "transect"
#TOPVIEW_DIR.mkdir(parents=True, exist_ok=True)
#TRANSECT_DIR.mkdir(parents=True, exist_ok=True)

TOPVIEW_DIR  = f"{frames_root}/{variable}/topview"
TRANSECT_DIR = f"{frames_root}/{variable}/transect"
Path(TOPVIEW_DIR).mkdir(parents=True, exist_ok=True)
Path(TRANSECT_DIR).mkdir(parents=True, exist_ok=True)

# Map window
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Transect definition
INDEX_LON_TRANSECT = 152
LAT_START = 90
LAT_STOP  = 153

ELEVATION_VAR_CANDIDATES = ("elev", "eta", "eta_out", "elevation", "ssh", "zeta")
SURFACE_LAYER_INDEX = 10  # 10 for the top layer, and 0 for the bottom layer
BATHY_VAR_CANDIDATES = ("bathymetry",)


# topview frames

#spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
frame_counter = 0

for spinup in spinups:
    print(f"\n=== Processing spinup: {spinup} ===")

    spinup_dir = os.path.join(base_dir, spinup)
    nc_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path) as ds:

            # Fix curvilinear grid
            lon_da = ds.lonc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lat_da = ds.latc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lon = lon_da.values
            lat = lat_da.values

            # Extract variable
            data_var = ds[variable]
            if "level" in data_var.dims:
                #data_var = data_var.isel(level=-1) # surface
                data_var = data_var.isel(level=1) # bottom

            # Mask variable
            mask_da = ds["elev"].squeeze(drop=True)
            mask_time_dim = _find_time_dim(mask_da)
            mask_da = _drop_duplicate_time(mask_da, mask_time_dim)

            # Build transect
            #tran_lon = lon[LAT_START:LAT_STOP, INDEX_LON_TRANSECT]
            #tran_lat = lat[LAT_START:LAT_STOP, INDEX_LON_TRANSECT]
            #valid = np.isfinite(tran_lon) & np.isfinite(tran_lat)
            #tran_lon = tran_lon[valid]
            #tran_lat = tran_lat[valid]

            # Loop through time steps
            for t_idx in range(data_var.sizes["time"]):

                frame = data_var.isel(time=t_idx).values.astype(float)

                # Apply mask
                mask2d = mask_da.isel({mask_time_dim: t_idx}).values
                wet_mask = mask2d > -1.1869 # -9999 for dry cells
                frame[~wet_mask] = np.nan

                timestamp = str(ds.time.values[t_idx])[:10]  # YYYY-MM-DD

                # Plot
                fig = plt.figure(figsize=(8, 6))
                ax = plt.axes(projection=ccrs.Mercator())
                ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

                # Basemap
                ctx.add_basemap(
                    ax,
                    crs=ax.projection.to_string(),
                    source=ctx.providers.Esri.WorldShadedRelief,
                    zorder=1,
                )
                txt = ax.texts[-1]
                txt.set_position([0.99, 0.02])
                txt.set_ha("right")

                # Field
                mesh = ax.pcolormesh(
                    lon,
                    lat,
                    frame,
                    transform=ccrs.PlateCarree(),
                    cmap=CMAP,
                    vmin=VMIN,
                    vmax=VMAX,
                    shading="gouraud",
                    zorder=2,
                )

                # Transect
                #ax.plot(
                #    tran_lon,
                #    tran_lat,
                #    "--",
                #    color="black",
                #    linewidth=2,
                #    transform=ccrs.PlateCarree(),
                #    zorder=5,
                #)

                # Colorbar
                cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
                cbar.set_label(f"{variable} ({var_unit})", fontsize=12)

                ax.set_title(f"{label} | {spinup} | {timestamp}")

                # Save frame with:
                #   - global counter (for MP4)
                #   - date (for readability)
                frame_path = os.path.join(
                    TOPVIEW_DIR,
                    f"{variable}_{frame_counter:06d}_{timestamp}.png"
                )

                plt.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1

print("\nDone. Frames saved in:")
print(TOPVIEW_DIR)

In [ ]:
# cell-par
param_variable ="ESS"
variable = param_variable

In [ ]:
# cell-2-mp4

# helper ----
def export_mp4_from_frames(frame_dir,
                           output_path,
                           fps=40): 

    frame_dir = Path(frame_dir)
    output_path = Path(output_path)

    frame_files = sorted(frame_dir.rglob("*.png"))
    if not frame_files:
        print(f"No PNG frames found in {frame_dir}")
        return

    print(f"Creating MP4 animation → {output_path}")
    print(f"Found {len(frame_files)} frames")

    writer = animation.FFMpegWriter(
        fps=fps,
        codec="libx264",
        bitrate=1800,
        extra_args=["-pix_fmt", "yuv420p"]   # <-- Windows compatible
    )

    first_img = plt.imread(frame_files[0])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(first_img)
    ax.axis("off")

    with writer.saving(fig, str(output_path), dpi=200):
        for f in frame_files:
            img = plt.imread(f)
            im.set_data(img)
            writer.grab_frame()

    plt.close(fig)
    print(f"MP4 saved: {output_path}")

# export the transect frames to MP4 to topview directory
export_mp4_from_frames(
    TOPVIEW_DIR,
    output_path= Path(frames_root) / variable / f"{variable}_topview.mp4",
    fps=40
)